### 모델 - 03. LLM 답변 캐싱하기 : page 201
- 답변 캐싱(Answer Caching)은 시스템 performance, 비용, 안정성을 모두 확보하기 위한 핵심 최적화 요소
- 동일하거나 유사한 질문에 대해 미리 저장된 결과를 반환함으로써 LLM API 비용을 0으로 만듬.
- 응답 속도(Latency) 획기적 개선
- 벡터 DB 검색 오버헤드 축소: 조회가 많을 경우.
- 답변 일관성 확보 : 유사한 질문에 대한 답변.

In [ ]:
# !pip --version

pip 25.0.1 from D:\skc0902\ex0917\.venv\Lib\site-packages\pip (python 3.12)



In [4]:
# !pip install dotenv
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

True

In [ ]:
# !pip install langchain langchain_openai
# !pip install langchain_community

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.globals import set_llm_cache
# 삭제될 예정 경고 뜸.
from langchain_core.caches import InMemoryCache
from langchain_community.cache import SQLiteCache
import os

C:\Users\user\AppData\Local\Temp\ipykernel_10572\1071808159.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import SQLiteCache


In [26]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

prompt = PromptTemplate.from_template("{country} 에 대해서 200자 내외로 요약해줘")

chain = prompt | llm

In [33]:
%%time 
response = chain.invoke({"country": "한국"})
print(response.content)
# 셀 명령어 : 셀(Cell) 전체의 실행 시간을 측정하는 셀 매직 명령어(Cell Magic Command)

한국은 동아시아에 위치한 고도로 발전한 현대화된 나라이다. 경제적으로 세계에서 선진국에 속하며 기술과 문화 분야에서도 선진화되어 있다. 전통문화와 현대문화가 공존하며 한류 열풍으로 세계에서 큰 인기를 얻고 있다. 또한 전통적인 음식과 아름다운 자연경관, 다양한 역사적 유적지 등이 매력적이다. 수도는 서울이며 국회의사당, 경복궁, 인사동 등이 대표적인 관광명소이다. 한반도 분단 상황이 계속되고 있지만, 평화정책과 국제사회와의 협력을 통해 평화를 향한 노력을 기울이고 있다.
CPU times: total: 0 ns
Wall time: 2.16 ms


In [39]:
%%time

# 인메모리 캐시 사용
set_llm_cache(InMemoryCache())

response2 = chain.invoke({"country": "한국"})
print(response2.content)

한국은 동아시아에 위치한 대한민국과 조선민주주의인민공화국으로 나뉘어 있다. 서울은 대한민국의 수도이며 발전된 경제와 현대화된 도시들이 많이 위치하고 있다. 역사적으로는 다양한 외국 문화의 영향을 받았으며, 고대부터 현대까지 다양한 문화와 전통이 이어져 왔다. 한국은 한류 열풍과 고급 IT산업 등으로 세계적인 영향력을 키우고 있으며, 한글은 한국어를 표기하는데 사용되는 문자이다. 한국은 높은 교육수준과 신속한 혁신으로 세계적으로 주목받고 있는 나라로, 전통과 현대가 공존하는 독특한 매력을 가지고 있다.
CPU times: total: 31.2 ms
Wall time: 2.25 s


### SQLite Cache : page 204
- 캐시(Cache) 레이어로 활용 : 질문(Query)의 임베딩 벡터나 텍스트 해시값을 키(Key)로, 생성된 LLM 답변 및 참고 문서 메타데이터를 값(Value)으로 저장하는 역학을 수행
- 빠름
- 영속성 : 순수 인메모리(In-Memory) 캐시와 달리 서버가 재시작되어도 캐시된 답변 데이터가 유실되지 않고 유지됨.

In [14]:
# 캐시 디렉토리 생성
if not os.path.exists("cache"):
    os.makedirs("cache")

# SQLiteCache를 사용
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))

In [53]:
%%time 
# 여러번 실행해보면, 저장된 캐시를 읽어옴(빠름, 내용같음).
response3 = chain.invoke({"country": "한국"})
print(response3.content)

한국은 동아시아에 위치한 대한민국과 조선민주주의인민공화국으로 나뉘어 있다. 서울은 대한민국의 수도이며 발전된 경제와 현대화된 도시들이 많이 위치하고 있다. 역사적으로는 다양한 외국 문화의 영향을 받았으며, 고대부터 현대까지 다양한 문화와 전통이 이어져 왔다. 한국은 한류 열풍과 고급 IT산업 등으로 세계적인 영향력을 키우고 있으며, 한글은 한국어를 표기하는데 사용되는 문자이다. 한국은 높은 교육수준과 신속한 혁신으로 세계적으로 주목받고 있는 나라로, 전통과 현대가 공존하는 독특한 매력을 가지고 있다.
CPU times: total: 0 ns
Wall time: 2.18 ms
